In [ ]:
import numpy as np
import torch
import time

In [32]:
import time
import numpy as np
import torch


def benchmark_numpy(matrix_a, matrix_b):
    start = time.perf_counter()

    result = matrix_a @ matrix_b

    end = time.perf_counter()

    return result, end - start


def benchmark_pytorch_mps(matrix_a, matrix_b):
    mps_device = torch.device("mps")

    tensor_a = torch.from_numpy(matrix_a).float().to(mps_device)
    tensor_b = torch.from_numpy(matrix_b).float().to(mps_device)

    # Warm-up
    _ = tensor_a @ tensor_b
    torch.mps.synchronize()

    start = time.perf_counter()

    result = tensor_a @ tensor_b

    # MPS operations are asynchronous
    torch.mps.synchronize()

    end = time.perf_counter()

    return result, end - start


# Example
n = 5000
matrix_a = np.random.rand(n, n).astype(np.float32)
matrix_b = np.random.rand(n, n).astype(np.float32)

numpy_result, numpy_time = benchmark_numpy(matrix_a, matrix_b)

pytorch_result, pytorch_time = benchmark_pytorch_mps(
    matrix_a,
    matrix_b
)

print(f"NumPy CPU:       {numpy_time:.6f} seconds")
print(f"PyTorch MPS:     {pytorch_time:.6f} seconds")
print(f"Speedup:         {numpy_time / pytorch_time:.2f}x")

NumPy CPU:       0.433629 seconds
PyTorch MPS:     0.061575 seconds
Speedup:         7.04x


In [12]:
def pd_matrix(rows, cols):
    matrix = np.random.randint(-10, 10, (rows, cols))
    pd_matrix = (matrix.T @ matrix).astype(float)

    return pd_matrix

In [17]:
def bench_compression(matrix):

    time_results = {}

    start_time = time.time()
    q, r = np.linalg.qr(matrix)
    end_time = time.time()

    print(f'Numpy execution time: {end_time - start_time}')
    qr = q @ r
    equal = np.isclose(qr, matrix).all()

    if equal:
        print(f'Numpy corectness: {equal}')

    time_results['numpy'] = [start_time, end_time, (end_time - start_time)]

    mps_device = torch.device('mps') if torch.mps.is_available() else torch.device('cpu')
    tensor_matrix = torch.from_numpy(matrix).float().to(mps_device)

    start_time = time.time()
    q, r = torch.linalg.qr(tensor_matrix)
    end_time = time.time()

    print(f'PyTorch execution time: {end_time - start_time}')
    qr = q @ r
    equal = np.isclose(qr, matrix).all()

    if equal:
        print(f'PyTorch corectness: {equal}')

    time_results['torch'] = [start_time, end_time, (end_time - start_time)]

    ratio1 = time_results['torch'][2] / time_results['numpy'][2]

    print(f'''
    The difference between PyTorch and Numpy is {ratio1} times
    ''')

In [18]:
matrix_1 = pd_matrix(5, 5)
bench_compression(matrix_1)

Numpy execution time: 0.00014281272888183594
Numpy corectness: True


NotImplementedError: The operator 'aten::linalg_qr.out' is not currently implemented for the MPS device. If you want this op to be added in priority during the prototype phase of this feature, please comment on https://github.com/pytorch/pytorch/issues/77764. As a temporary fix, you can set the environment variable `PYTORCH_ENABLE_MPS_FALLBACK=1` to use the CPU as a fallback for this op. WARNING: this will be slower than running natively on MPS.

In [5]:
matrix_1 = pd_matrix(10, 10)
bench_compression(matrix_1)

Numpy execution time: 0.00015616416931152344
Numpy corectness: True
PyTorch execution time: 0.0001971721649169922
PyTorch corectness: True

    The difference between PyTorch and Numpy is 1.2625954198473281 times
    


In [6]:
matrix_1 = pd_matrix(20, 20)
bench_compression(matrix_1)

Numpy execution time: 0.000125885009765625
Numpy corectness: True
PyTorch execution time: 0.0002071857452392578
PyTorch corectness: True

    The difference between PyTorch and Numpy is 1.6458333333333333 times
    


In [7]:
matrix_1 = pd_matrix(50, 50)
bench_compression(matrix_1)

Numpy execution time: 0.00038504600524902344
Numpy corectness: True
PyTorch execution time: 0.009868144989013672
PyTorch corectness: True

    The difference between PyTorch and Numpy is 25.628482972136222 times
    


In [8]:
matrix_1 = pd_matrix(100, 100)
bench_compression(matrix_1)

Numpy execution time: 0.0014100074768066406
Numpy corectness: True
PyTorch execution time: 0.00045609474182128906
PyTorch corectness: True

    The difference between PyTorch and Numpy is 0.32346973283733516 times
    


In [9]:
matrix_1 = pd_matrix(200, 200)
bench_compression(matrix_1)

Numpy execution time: 0.05549478530883789
Numpy corectness: True
PyTorch execution time: 0.002766847610473633
PyTorch corectness: True

    The difference between PyTorch and Numpy is 0.0498577946572035 times
    


In [10]:
matrix_1 = pd_matrix(500, 500)
bench_compression(matrix_1)

Numpy execution time: 0.03320026397705078
Numpy corectness: True
PyTorch execution time: 0.012186050415039062
PyTorch corectness: True

    The difference between PyTorch and Numpy is 0.36704679286473446 times
    
